In [1]:
print("Let's Start")

Let's Start


GRU - GAted Recurrent Unit is also advance version of RNN , which have abilities of LSTM with less computational weights...

In [2]:
# Project USing LSTM on next word Predictor from chats Whatsapp chats DATASET

import pandas as pd

df1 = pd.read_csv(r"C:\Users\Abhishek sharma\OneDrive\ドキュメント\waste\ML csv data\hackothan.csv")
# df2 = pd.read_csv(r"C:\Users\Abhishek sharma\OneDrive\ドキュメント\waste\ML csv data\groupchat.csv")
df2 = pd.read_csv(r"C:\Users\Abhishek sharma\OneDrive\ドキュメント\waste\ML csv data\chat_With_yash.csv")
df = pd.concat([df1, df2], axis=0)

In [3]:
df = df['message']
df.head()

0              Hllo I got ur number from naitik Jangra
1                           Tell me ur course and year
2                                                   Ha
3                                   Btech AIML Vth sem
4    Ok I want to know ki aap pahli baar le rahe h ...
Name: message, dtype: object

In [4]:
import re
def remove_emoji(text):
    if not isinstance(text, str):
        return text

    emoji_pattern = re.compile(
        "["

        # emoticons
        "\U0001F600-\U0001F64F"

        # symbols & pictographs
        "\U0001F300-\U0001F5FF"

        # transport & map symbols
        "\U0001F680-\U0001F6FF"

        # flags (iOS)
        "\U0001F1E0-\U0001F1FF"

        # dingbats
        "\U00002700-\U000027BF"

        # miscellaneous symbols
        "\U00002600-\U000026FF"

        # enclosed characters / other pictographs
        "\U000024C2-\U0001F251"

        # supplemental symbols & pictographs (covers 🥰, 🥱, 🤧, 🤦 etc.)
        "\U0001F900-\U0001F9FF"

        # Symbols & Pictographs Extended-A (newer emojis)
        "\U0001FA70-\U0001FAFF"

        # some other supplemental ranges (optional)
        "\U0001F700-\U0001F77F"

        "]+",
        flags=re.UNICODE
    )
    text = emoji_pattern.sub("", text)
    text = re.sub(r"[\uFE0F\u200D]", "", text)

    return text

df = df.apply(remove_emoji)
df.head()

0              Hllo I got ur number from naitik Jangra
1                           Tell me ur course and year
2                                                   Ha
3                                   Btech AIML Vth sem
4    Ok I want to know ki aap pahli baar le rahe h ...
Name: message, dtype: object

In [5]:
import token
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer

tokens = Tokenizer()
tokens.fit_on_texts(df)

df = df.apply(lambda x : tokens.texts_to_sequences([x]))
df.head()

0        [[1002, 210, 664, 418, 351, 665, 1003, 1004]]
1                      [[666, 20, 418, 1005, 98, 145]]
2                                               [[19]]
3                              [[667, 668, 1006, 669]]
4    [[28, 210, 670, 2, 503, 5, 119, 504, 92, 69, 1...
Name: message, dtype: object

In [6]:
df = df.values
type(df)

numpy.ndarray

In [7]:
df

array([list([[1002, 210, 664, 418, 351, 665, 1003, 1004]]),
       list([[666, 20, 418, 1005, 98, 145]]), list([[19]]), ...,
       list([[646, 647, 21, 939, 221, 10, 9, 247, 160, 2017, 2018, 6, 2019, 17, 81, 2020, 128, 10]]),
       list([[]]), list([[19]])], dtype=object)

In [8]:
input_seq = []
for i in df:
    for j in i:
        for k in range(1,len(j)):
            n_grams = j[:k+1]
            input_seq.append(n_grams)

In [9]:
len_list = []
for i in input_seq:
    len_list.append(len(i))

max_len = max(len_list)

In [10]:
from keras.preprocessing.sequence import pad_sequences
input_sequences = pad_sequences(input_seq , padding='pre' , maxlen= max_len)
input_sequences


array([[   0,    0,    0, ...,    0, 1002,  210],
       [   0,    0,    0, ..., 1002,  210,  664],
       [   0,    0,    0, ...,  210,  664,  418],
       ...,
       [   0,    0,    0, ...,   17,   81, 2020],
       [   0,    0,    0, ...,   81, 2020,  128],
       [   0,    0,    0, ..., 2020,  128,   10]])

In [11]:
y = input_sequences[:,-1]
x = input_sequences[:,:-1]


In [12]:
y.shape

(8819,)

In [13]:

x.shape

(8819, 221)

In [14]:
from keras.utils import to_categorical
y =to_categorical(y , num_classes=2021)
y.shape

(8819, 2021)

In [17]:
vocab_size = len(tokens.word_index) + 1
vocab_size

2021

In [18]:
from keras.models import Sequential
from keras.layers import Input, Embedding, GRU, Dense, Dropout, Bidirectional

model = Sequential([
    Input(shape=(220,)),   # 🔥 THIS FIXES EVERYTHING
    Embedding(input_dim=2021, output_dim=100),
    Bidirectional(GRU(128)),
    Dropout(0.5),
    Dense(2021, activation='softmax')
])

model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.summary()



Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 220, 100)       │       202,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 256)            │       176,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2021)           │       519,397 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 898,137 (3.43 MB)

 Trainable params: 898,137 (3.43 MB)

 Non-trainable params: 0 (0.00 B)

In [19]:
print(type(x))
print(type(y))
print(x.shape)
print(y.shape)


<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
(8819, 221)
(8819, 2021)


In [20]:
model.fit(x , y , epochs=10)

Epoch 1/10
276/276 ━━━━━━━━━━━━━━━━━━━━ 65s 210ms/step - accuracy: 0.0497 - loss: 6.9638
Epoch 2/10
276/276 ━━━━━━━━━━━━━━━━━━━━ 58s 209ms/step - accuracy: 0.0603 - loss: 6.3272
Epoch 3/10
276/276 ━━━━━━━━━━━━━━━━━━━━ 55s 201ms/step - accuracy: 0.0797 - loss: 5.9774
Epoch 4/10
276/276 ━━━━━━━━━━━━━━━━━━━━ 56s 201ms/step - accuracy: 0.0945 - loss: 5.7130
Epoch 5/10
276/276 ━━━━━━━━━━━━━━━━━━━━ 58s 210ms/step - accuracy: 0.1165 - loss: 5.4019
Epoch 6/10
276/276 ━━━━━━━━━━━━━━━━━━━━ 57s 206ms/step - accuracy: 0.1309 - loss: 5.1467
Epoch 7/10
276/276 ━━━━━━━━━━━━━━━━━━━━ 58s 209ms/step - accuracy: 0.1637 - loss: 4.8254
Epoch 8/10
276/276 ━━━━━━━━━━━━━━━━━━━━ 59s 213ms/step - accuracy: 0.1902 - loss: 4.5149
Epoch 9/10
276/276 ━━━━━━━━━━━━━━━━━━━━ 60s 216ms/step - accuracy: 0.2214 - loss: 4.2482
Epoch 10/10
276/276 ━━━━━━━━━━━━━━━━━━━━ 62s 225ms/step - accuracy: 0.2452 - loss: 3.9963


In [24]:
import numpy as np


def predictor(text = ""):
    for i in range(10):
        tokenised_text = tokens.texts_to_sequences([text])[0]
        padded_text = pad_sequences([tokenised_text] , maxlen=max_len , padding='pre')
        pos = np.argmax(model.predict(padded_text))
        for word , index in tokens.word_index.items():
            if index==pos:
                text = text + " " + word
                print(text)

predictor(text = "chut")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step
chut mt
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
chut mt kr
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
chut mt kr rha
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
chut mt kr rha hu
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
chut mt kr rha hu to
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step
chut mt kr rha hu to ye
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
chut mt kr rha hu to ye bhi
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
chut mt kr rha hu to ye bhi badi
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
chut mt kr rha hu to ye bhi badi problem
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step
chut mt kr rha hu to ye bhi badi problem nahi
